# Oficina: seu mapa da ciência no Colab

Neste caderno você monta, do zero, um mapa dos artigos da revista *Opinião Pública* de 2020 a 2024:

1. coleta os títulos e resumos no SciELO e no OpenAlex;
2. agrupa os artigos em tópicos, com um modelo de *embeddings*;
3. descobre de onde vêm os autores;
4. classifica uma amostra de resumos com um modelo de linguagem, que cita o trecho que justifica cada resposta;
5. abre o painel, com o mapa, os tópicos no tempo, a geografia e a classificação.

Tudo roda nesta máquina do Colab, com modelos abertos, sem chave de API e sem mandar os textos para ninguém. Leva uns 30 minutos, a maior parte com os modelos trabalhando sozinhos.

**Antes de começar:** no menu, *Ambiente de execução › Alterar o tipo de ambiente de execução*, escolha **GPU T4**. Depois rode as células em ordem (Shift+Enter).

O que cada etapa faz está na [documentação](https://felipelamarca.com/mapa-da-ciencia/), e o roteiro da oficina, em [Oficina no Colab](https://felipelamarca.com/mapa-da-ciencia/tutoriais/oficina-colab/).

## 1. Instalar

O `mapa-da-ciencia` e o [Ollama](https://ollama.com), que roda os modelos. Uns 2 minutos.

In [ ]:
VERSAO = "2.1.0"  # a versão do mapa-da-ciencia desta oficina

!nvidia-smi -L || echo "Sem GPU: escolha a T4 em Ambiente de execução › Alterar o tipo de ambiente de execução."
!pip install -q "https://github.com/felipelmc/mapa-da-ciencia/releases/download/v{VERSAO}/mapa_da_ciencia-{VERSAO}-py3-none-any.whl"
!mapa --versao

In [ ]:
!apt-get -qq install -y zstd > /dev/null  # o instalador do Ollama precisa dele
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

ollama = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

## 2. Baixar os modelos

Dois modelos abertos: um de *embeddings* (0,6 GB), que transforma cada resumo num vetor, e um de linguagem (6,6 GB), que escreve os rótulos dos tópicos e classifica os resumos. Uns 3 minutos.

In [ ]:
!ollama pull qwen3-embedding:0.6b
!ollama pull qwen3.5:9b

## 3. Criar o projeto e coletar

Um projeto é uma pasta com a configuração (`mapa.yaml`), o codebook (`codebook.yaml`) e os dados. O recorte: a *Opinião Pública*, de 2020 a 2024, com o perfil de modelos `padrao`, o que cabe na T4. Para ver os arquivos, abra o painel de arquivos do Colab (o ícone de pasta, à esquerda) e dê um clique duplo em `oficina/mapa.yaml`.

In [ ]:
%cd /content
!mapa novo oficina --revista op --anos 2020-2024 --perfil padrao
%cd oficina
!mapa coletar

## 4. Tópicos e geografia

`mapa topicos` calcula os *embeddings*, agrupa os artigos e pede ao modelo um rótulo em português para cada tópico. `mapa geografia` liga as afiliações dos autores às instituições e conta a produção por UF e país.

In [ ]:
!mapa topicos
!mapa geografia

## 5. Classificar uma amostra

O `codebook.yaml` pergunta a abordagem, a técnica, o recorte geográfico, a subárea e o período de cada artigo. Sorteamos 30 artigos, e o modelo lê cada um e responde com a evidência de cada resposta. Uns 6 minutos. (No seu computador, `mapa classificar` sem opções classifica o corpus inteiro.)

In [ ]:
!mapa validar amostra --n 30
!mapa classificar --somente-amostra
!mapa status

## 6. Abrir o painel

O painel abre numa janela nova, pelo *proxy* do Colab. Explore o **Mapa** (cada ponto é um artigo), os **Tópicos** no tempo, a **Geografia** e a **Classificação**. Em **Validação › Codificar a amostra**, você mesmo classifica os 30 artigos, às cegas, e vê quanto concorda com o modelo.

A célula termina logo, e o painel continua no ar enquanto o caderno estiver aberto. Se a janela não abrir, libere as janelas *pop-up* do Colab e rode a célula de novo.

In [ ]:
import mapa_da_ciencia.api as mapa

mapa.painel(".")

## Para ir além

- Troque a revista ou o período em `!mapa novo` (`!mapa revistas "ciência política"` lista as revistas).
- Edite o `codebook.yaml` (no painel de arquivos, à esquerda) com as suas perguntas e rode `!mapa classificar --somente-amostra` de novo.
- Leve o projeto: `!zip -r oficina.zip . -x "brutos/*"` e baixe o zip pelo painel de arquivos. A máquina do Colab é apagada quando a sessão termina.
- No seu computador, siga o tutorial [Seu primeiro mapa](https://felipelamarca.com/mapa-da-ciencia/tutoriais/primeiro-mapa/).